In [1]:
#loading filtered rna data
import anndata
from anndata import AnnData

#RNA data with only the filtered cells, still have "unnormalized" counts
filtered_Rdata = anndata.read_h5ad("/home/fgsasse_lrs_1/Downloads/BA/BA_data/RNA/filtered_Rdata.h5ad")
filtered_Rdata.obs

,developmental_stage,dataset,zebrafish_anatomy_ontology_class,zebrafish_anatomy_ontology_class_coarse,timepoint,n_genes_by_counts,total_counts,total_counts_mt,pct_counts_mt,total_counts_nc,pct_counts_nc
AAACAGCCACCTAAGC-1_1,15somites,TDR118,epidermis,epidermis,16hpf,2317,6522.0,160.0,2.453235,726.0,11.131555
AAACAGCCAGGGAGGA-1_1,15somites,TDR118,pronephros,pronephros,16hpf,2319,6100.0,245.0,4.016393,1051.0,17.229508
AAACAGCCATAGACCC-1_1,15somites,TDR118,hindbrain,hindbrain,16hpf,3467,12581.0,779.0,6.191877,2542.0,20.205071
AAACATGCAAACTCAT-1_1,15somites,TDR118,spinal_cord,spinal_cord,16hpf,2145,5642.0,265.0,4.696916,1075.0,19.053527
AAACATGCAAGGACCA-1_1,15somites,TDR118,neural_optic2,neural_optic,16hpf,838,2691.0,181.0,6.726124,732.0,27.201784
...,...,...,...,...,...,...,...,...,...,...,...
TTTGTGTTCCCTCAGT-1_7,10somites,TDR128,tail_bud,tail_bud,14hpf,1281,3079.0,114.0,3.702501,572.0,18.577460
TTTGTTGGTACCTTAC-1_7,10somites,TDR128,lateral_plate_mesoderm,lateral_plate_mesoderm,14hpf,2441,6276.0,179.0,2.852135,962.0,15.328235
TTTGTTGGTATTGAGT-1_7,10somites,TDR128,neural_posterior,neural_posterior,14hpf,1164,2377.0,74.0,3.113168,217.0,9.129154
TTTGTTGGTGCGCGTA-1_7,10somites,TDR128,endocrine_pancreas,endocrine_pancreas,14hpf,811,1750.0,54.0,3.085714,182.0,10.400000


In [2]:
#check the expression matrix of the filtered RNA data, if the counts are still unnormalized
import numpy as np
rna_matrix = filtered_Rdata.layers["counts"][:,:]
print("min:", rna_matrix.min(), "max:", rna_matrix.max())

#check if all non-zero values are integers
non_zero_values = rna_matrix[rna_matrix != 0]
if np.all(non_zero_values.astype(int) == non_zero_values):
    print("All non-zero values are integers.")


min: 0.0 max: 8576.0
All non-zero values are integers.


In [3]:
print("Max percentage of MT genes in RNA data:", filtered_Rdata.obs["pct_counts_mt"].max())
print("Min percentage of MT genes in RNA data:", filtered_Rdata.obs["pct_counts_mt"].min())
#cheking how many cells have more than 10% of MT genes
num_cells_high_mt = (filtered_Rdata.obs["pct_counts_mt"] > 10).sum()
print("Number of cells with more than 10% MT genes:", num_cells_high_mt)    

Max percentage of MT genes in RNA data: 16.584500098599882
Min percentage of MT genes in RNA data: 0.11827321111768185
Number of cells with more than 10% MT genes: 223


## Pseudobulking the RNA-Seq data
- creating new matrix of raw counts of RNA-Seq data 
- creating df for metadata 
- aggregating the counts by groups 1) cell type, 2) cell type and developmental stages
    - aggregation equation: summing raw counts per gene across all cells within one group
    - aggregation normalization: total count per gene in group / total count across all genes in that group
This normalizes each gene's count by the total counts (across all genes) within each cell type group, giving a proportional representation rather than raw counts.

In [4]:
#creating a aggregation function to sum the counts for each group then divide by the total counts
def agg_fun(mat, meta, fun="sum", gene_names=None):
    """Aggregate count matrix by metadata groups (e.g., cell type + developmental time)."""
    import pandas as pd
    import numpy as np
    
    meta = pd.Series(meta)
    
    if fun == "sum":
        def agg_f(x):
            col_sums = mat[x, :].sum(axis=0)  # Sum counts for each column (gene) in selected rows
            total = mat[x, :].sum()           # Total sum of counts in selected rows
            return col_sums / total
        
        # Apply aggregation function to each group
        groups = meta.unique()
        agg_result = pd.DataFrame(
            [agg_f(meta == group) for group in groups],
            index=groups,
            columns=gene_names if gene_names is not None else None,
        )
        return agg_result

In [5]:
# Convert sparse matrix to dense if needed & extract metadata
rna_ct_matrix = filtered_Rdata.layers["counts"][:,:].toarray()  # Convert sparse matrix to dense if it's sparse
cell_types = filtered_Rdata.obs["zebrafish_anatomy_ontology_class"].astype(str)
dev_times = filtered_Rdata.obs["developmental_stage"].astype(str)
gene_names = filtered_Rdata.var_names




In [6]:
#Aggregate by cell type
agg_rna_ct = agg_fun(rna_ct_matrix, cell_types, fun="sum", gene_names=gene_names)
agg_rna_ct.head()


,ptpn12,phtf2,phtf2.1,CU856344.1,si:zfos-932h1.3,mansc1,lrp6,dusp16,crebl2,gpr19,...,mt-nd4,NC-002333.16,NC-002333.15,NC-002333.8,mt-nd5,mt-nd6,NC-002333.21,mt-cyb,NC-002333.22,NC-002333.11
epidermis,1.598638e-06,7.645658e-07,9.730838e-07,6.950599e-08,6.950599e-07,0.000000e+00,0.000003,0.000007,0.000005,0.000006,...,0.001352,0.000012,0.000007,1.390120e-07,0.000376,0.000077,2.085180e-07,0.004257,4.170359e-07,2.085180e-07
pronephros,6.582024e-07,4.701446e-07,6.582024e-07,9.402891e-08,1.598491e-06,9.402891e-08,0.000003,0.000007,0.000004,0.000008,...,0.001250,0.000012,0.000007,0.000000e+00,0.000374,0.000069,6.582024e-07,0.004251,6.582024e-07,5.641735e-07
hindbrain,7.688362e-07,7.688362e-07,2.050230e-06,8.542624e-08,8.542624e-07,0.000000e+00,0.000004,0.000006,0.000006,0.000004,...,0.001372,0.000009,0.000006,1.708525e-07,0.000332,0.000076,3.417050e-07,0.004376,6.834099e-07,5.125574e-07
spinal_cord,1.136235e-06,7.866242e-07,2.053963e-06,1.311040e-07,1.048832e-06,0.000000e+00,0.000003,0.000005,0.000005,0.000005,...,0.001345,0.000008,0.000006,1.311040e-07,0.000358,0.000077,1.748054e-07,0.004328,4.370134e-07,3.059094e-07
neural_optic2,8.039956e-07,9.044951e-07,2.060239e-06,5.024973e-08,1.457242e-06,0.000000e+00,0.000003,0.000006,0.000004,0.000008,...,0.001228,0.000010,0.000007,2.009989e-07,0.000360,0.000078,6.029967e-07,0.004112,5.024973e-07,3.517481e-07


In [7]:
#Aggregate by cell type and developmental stages
meta_ct_time = cell_types + "__" + dev_times #Build modified meta vector: cell type + developmental time

agg_rna_ct_time = agg_fun(rna_ct_matrix, meta_ct_time, fun="sum", gene_names=gene_names)
agg_rna_ct_time.head()

,ptpn12,phtf2,phtf2.1,CU856344.1,si:zfos-932h1.3,mansc1,lrp6,dusp16,crebl2,gpr19,...,mt-nd4,NC-002333.16,NC-002333.15,NC-002333.8,mt-nd5,mt-nd6,NC-002333.21,mt-cyb,NC-002333.22,NC-002333.11
epidermis__15somites,1.702130e-06,6.382988e-07,1.489364e-06,2.127663e-07,6.382988e-07,0.0,0.000004,0.000007,0.000004,1.702130e-06,...,0.001350,0.000008,0.000011,2.127663e-07,0.000390,0.000125,4.255325e-07,0.003727,4.255325e-07,0.000000e+00
pronephros__15somites,1.875782e-06,4.689455e-07,9.378910e-07,0.000000e+00,1.875782e-06,0.0,0.000008,0.000005,0.000004,9.378910e-07,...,0.001321,0.000007,0.000015,0.000000e+00,0.000422,0.000132,4.689455e-07,0.003901,1.406836e-06,4.689455e-07
hindbrain__15somites,8.519315e-07,1.135909e-06,3.123749e-06,2.839772e-07,8.519315e-07,0.0,0.000005,0.000007,0.000003,8.519315e-07,...,0.001379,0.000006,0.000011,0.000000e+00,0.000430,0.000137,8.519315e-07,0.004009,8.519315e-07,1.419886e-06
spinal_cord__15somites,1.921197e-06,1.280798e-06,3.201994e-06,1.280798e-07,1.665037e-06,0.0,0.000003,0.000006,0.000005,3.842393e-07,...,0.001368,0.000004,0.000012,0.000000e+00,0.000418,0.000129,5.123191e-07,0.003953,3.842393e-07,3.842393e-07
neural_optic2__15somites,1.049168e-06,1.398890e-06,3.322364e-06,0.000000e+00,2.273197e-06,0.0,0.000004,0.000006,0.000003,1.049168e-06,...,0.001305,0.000008,0.000014,0.000000e+00,0.000412,0.000147,1.224029e-06,0.003909,5.245838e-07,5.245838e-07


In [8]:
# Checking the aggregation results by the shape of the resulting dataframe and its columns
print(agg_rna_ct.shape)
print(agg_rna_ct.index)
agg_rna_ct.columns

print(agg_rna_ct_time.shape)
print(agg_rna_ct_time.index)
agg_rna_ct_time.columns

(39, 32057)
Index(['epidermis', 'pronephros', 'hindbrain', 'spinal_cord', 'neural_optic2',
       'neural_floor_plate', 'neural_crest2', 'PSM', 'optic_cup',
       'lateral_plate_mesoderm', 'midbrain_hindbrain_boundary2',
       'neural_telencephalon', 'differentiating_neurons', 'muscle',
       'fast_muscle', 'heart_myocardium', 'somites', 'NMPs', 'epidermis2',
       'pharyngeal_arches', 'floor_plate2', 'hemangioblasts',
       'neural_posterior', 'floor_plate', 'tail_bud', 'endoderm',
       'midbrain_hindbrain_boundary', 'neural_crest', 'neural_optic',
       'hematopoietic_vasculature', 'endocrine_pancreas', 'hatching_gland',
       'neurons', 'notochord', 'pronephros2', 'enteric_neurons', 'epidermis3',
       'epidermis4', 'neural'],
      dtype='object')
(196, 32057)
Index(['epidermis__15somites', 'pronephros__15somites', 'hindbrain__15somites',
       'spinal_cord__15somites', 'neural_optic2__15somites',
       'neural_floor_plate__15somites', 'neural_crest2__15somites',
      

Index(['ptpn12', 'phtf2', 'phtf2.1', 'CU856344.1', 'si:zfos-932h1.3', 'mansc1',
       'lrp6', 'dusp16', 'crebl2', 'gpr19',
       ...
       'mt-nd4', 'NC-002333.16', 'NC-002333.15', 'NC-002333.8', 'mt-nd5',
       'mt-nd6', 'NC-002333.21', 'mt-cyb', 'NC-002333.22', 'NC-002333.11'],
      dtype='object', length=32057)

In [33]:
#Comparing the number of combinations of cell types and developmental stages ("meta_ct_time") with the index (row) of the aggregated data frame to find out which groups are (not) present in the aggregated data frame
import numpy as np

meta_groups = set(meta_ct_time.unique())
agg_groups = set(agg_rna_ct_time.index)

print("Missing in ct + time dataframe:", meta_groups - agg_groups)
print("Number of combinationsof cell types and developmental stages:", len(meta_groups))
print("Number of groups in the ct + time dataframe:", len(agg_groups))


Missing in ct + time dataframe: set()
Number of combinationsof cell types and developmental stages: 196
Number of groups in the ct + time dataframe: 196


In [9]:
# Save the aggregated data frames to a new AnnData object
import pandas as pd
agg_rna_ct = AnnData(X=agg_rna_ct.values, obs=pd.DataFrame(index=agg_rna_ct.index), var=pd.DataFrame(index=agg_rna_ct.columns))
agg_rna_ct.write_h5ad("/home/fgsasse_lrs_1/Downloads/BA/BA_data/Pseudobulks/RNA/celltypes/agg_rna_ct.h5ad")

agg_rna_ct_time = AnnData(X=agg_rna_ct_time.values, obs=pd.DataFrame(index=agg_rna_ct_time.index), var=pd.DataFrame(index=agg_rna_ct_time.columns))
agg_rna_ct_time.write_h5ad("/home/fgsasse_lrs_1/Downloads/BA/BA_data/Pseudobulks/RNA/celltypes_times/agg_rna_ct_time.h5ad")

## Log-transformation?
There's no need to log-transform due to the fact, that the values are proportions (0 to 1) 